# lensless reconstruction demo

paste a google drive zip link in the config cell below, run all, reconstructions land in `OUTPUT_DIR`.
i run my best model here — leaddm5 + drunet pre+post (~17.3 db on digicam test).

supported zip layout (same as `CustomDirDataset`):

```
dataset/
  lensless/
    ImageID.png
  masks/
    ImageID.npy
  lensed/
    ImageID.png   # optional, only needed for visual gt + metrics
```

i expect `lensless/*.png` and `masks/*.npy` with matching stems. `lensed/` is optional — without it inference still runs, i just skip metrics.

## setup

i clone my repo, install what i put in `requirements.txt`, and pull in `gdown` for drive zips. same stack as the README, minus the venv stuff colab doesn't need. safe to re-run — clone is skipped if the folder's already there.

In [ ]:
import os

REPO_URL = "https://github.com/tolyaho/lensless-computational-imaging.git"
REPO_DIR = "/content/lensless-computational-imaging"

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt
!pip install -q gdown

## input

only thing you need to change is `DATASET_ZIP_URL`. `OUTPUT_DIR` is optional — by default i save reconstructions to `/content/lensless_demo_outputs/reconstructions`.

In [ ]:
DATASET_ZIP_URL = "drop ur link here"

UNZIP_DIR = "/content/lensless_demo_data"
OUTPUT_DIR = "/content/lensless_demo_outputs/reconstructions"

MODEL_PRESET = "leadmm5_drunet8m_prepost"
NUM_VIS_SAMPLES = 5

print("output dir:", OUTPUT_DIR)

## checkpoints

i download only the default demo checkpoint (my best pre+post drunet model). no manual upload.

In [ ]:
from pathlib import Path

CHECKPOINT_PATH = Path("checkpoints/leadmm5_prepost_drunet8m.pth")

!python download_checkpoints.py --name leadmm5_prepost_drunet8m.pth

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"checkpoint not found after download: {CHECKPOINT_PATH}")

## data

i wipe old unzip/output dirs first so reruns don't mix datasets. then download your zip, unpack, and find the folder with `lensless/` + `masks/` inside.

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown


def find_dataset_root(base_dir):
    base_dir = Path(base_dir)
    candidates = [base_dir] + [p for p in base_dir.rglob("*") if p.is_dir()]
    for p in candidates:
        if (p / "lensless").is_dir() and (p / "masks").is_dir():
            return p
    raise FileNotFoundError("could not find dataset root with lensless/ and masks/")


def download_zip(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if "drive.google.com" in url or "docs.google.com" in url:
        gdown.download(url, str(dest), fuzzy=True)
        return

    try:
        import requests

        response = requests.get(url, timeout=120)
        response.raise_for_status()
        dest.write_bytes(response.content)
    except Exception:
        if shutil.which("wget"):
            subprocess.run(["wget", "-O", str(dest), url], check=True)
        else:
            raise


if "PASTE_GOOGLE_DRIVE" in DATASET_ZIP_URL:
    raise ValueError("set DATASET_ZIP_URL in the input cell above")

unzip_dir = Path(UNZIP_DIR)
output_dir = Path(OUTPUT_DIR)
zip_path = Path(f"{UNZIP_DIR}.zip")

shutil.rmtree(unzip_dir, ignore_errors=True)
unzip_dir.mkdir(parents=True, exist_ok=True)

shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)

zip_path.unlink(missing_ok=True)
download_zip(DATASET_ZIP_URL, zip_path)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(unzip_dir)

DATASET_ROOT = find_dataset_root(unzip_dir)

lensless_dir = DATASET_ROOT / "lensless"
masks_dir = DATASET_ROOT / "masks"

print("dataset root:", DATASET_ROOT)
print("lensless files:", len(list(lensless_dir.glob("*.png"))))
print("masks files:", len(list(masks_dir.glob("*.npy"))))
print("raw lensed/ exists:", (DATASET_ROOT / "lensed").is_dir())

## inference

i run my best report model through `inference.py`. reconstructions go to `OUTPUT_DIR/recon`. if your zip had `lensed/`, inference also saves aligned gt to `OUTPUT_DIR/lensed` — i prefer that for viz + metrics.

In [ ]:
import shlex
import shutil
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)

dataset_root_arg = shlex.quote(str(DATASET_ROOT))
output_dir_arg = shlex.quote(str(OUTPUT_DIR))
checkpoint_arg = shlex.quote(str(CHECKPOINT_PATH))

!python inference.py -cn inference_custom \
    model=modular_leadmm5_prepost_drunet8m \
    checkpoint={checkpoint_arg} \
    dataset.root={dataset_root_arg} \
    output_dir={output_dir_arg} \
    save_targets=true

RECON_DIR = Path(OUTPUT_DIR) / "recon"
print("reconstructions saved to:", RECON_DIR)

saved_lensed_dir = Path(OUTPUT_DIR) / "lensed"
raw_lensed_dir = Path(DATASET_ROOT) / "lensed"

if saved_lensed_dir.is_dir() and any(saved_lensed_dir.iterdir()):
    LENSED_DIR = saved_lensed_dir
elif raw_lensed_dir.is_dir() and any(raw_lensed_dir.iterdir()):
    LENSED_DIR = raw_lensed_dir
else:
    LENSED_DIR = None

## samples

side-by-side plots matched by filename stem. inference expects `lensless/*.png`; display also handles `.jpg`/`.jpeg` if present. if aligned gt exists i show `original | lensless | recon`, otherwise just `lensless | recon`.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

IMAGE_EXTS = (".png", ".jpg", ".jpeg")


def index_images(folder: Path) -> dict[str, Path]:
    images = {}
    if not folder.is_dir():
        return images
    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
            images[path.stem] = path
    return images


def find_by_stem(folder: Path, stem: str) -> Path | None:
    if not folder.is_dir():
        return None
    for ext in IMAGE_EXTS:
        candidate = folder / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    return None


recon_dir = Path(OUTPUT_DIR) / "recon"
lensless_dir = DATASET_ROOT / "lensless"

recon_files = index_images(recon_dir)
if not recon_files:
    raise FileNotFoundError(f"no reconstructions found in {recon_dir}")

sample_ids = sorted(recon_files.keys())[:NUM_VIS_SAMPLES]

for stem in sample_ids:
    panels = []
    titles = []

    if LENSED_DIR is not None:
        lensed_path = find_by_stem(LENSED_DIR, stem)
        if lensed_path is not None:
            panels.append(Image.open(lensed_path).convert("RGB"))
            titles.append("original/aligned gt")

    lensless_path = find_by_stem(lensless_dir, stem)
    if lensless_path is None:
        png_path = lensless_dir / f"{stem}.png"
        raise FileNotFoundError(
            f"no lensless image for stem {stem!r}; inference expects lensless/*.png ({png_path})"
        )

    panels.append(Image.open(lensless_path).convert("RGB"))
    titles.append("lensless")

    panels.append(Image.open(recon_files[stem]).convert("RGB"))
    titles.append("reconstruction")

    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    if len(panels) == 1:
        axes = [axes]

    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(stem)
    plt.show()

print(f"showing {len(sample_ids)} / {len(recon_files)} reconstructions")

## metrics

if aligned/saved gt is available, i run the same `calculate_metrics.py` as the report (`use_roi=true`). otherwise i skip.

In [ ]:
import json
import shlex
from pathlib import Path

pred_dir = Path(OUTPUT_DIR) / "recon"
metrics_path = Path(OUTPUT_DIR) / "metrics.json"

if LENSED_DIR is None:
    print("no lensed/ folder found, metrics skipped")
else:
    pred_dir_arg = shlex.quote(str(pred_dir))
    gt_dir_arg = shlex.quote(str(LENSED_DIR))
    metrics_path_arg = shlex.quote(str(metrics_path))

    !python calculate_metrics.py pred_dir={pred_dir_arg} gt_dir={gt_dir_arg} output_path={metrics_path_arg} use_roi=true

    results = json.loads(metrics_path.read_text())
    for name in ("PSNR", "SSIM", "MSE", "LPIPS"):
        if name in results:
            print(f"{name}: {results[name]}")

## done

reconstructions are in `OUTPUT_DIR`. if gt was available, metrics are printed above too.

In [ ]:
import shlex
from pathlib import Path

recon_dir = Path(OUTPUT_DIR) / "recon"
metrics_path = Path(OUTPUT_DIR) / "metrics.json"

print("dataset root:", DATASET_ROOT)
print("reconstruction folder:", recon_dir)
print("lensed/gt folder:", LENSED_DIR)
print("metrics path:", metrics_path if metrics_path.is_file() else None)

In [ ]:
import shlex

recon_dir_arg = shlex.quote(str(recon_dir))
!zip -qr reconstructions.zip {recon_dir_arg}
print("saved reconstructions.zip")